# LMA Individual Project - Phase 2 pretraining (Kaggle)

Pretrains **one** ~25M-parameter decoder-only Transformer from scratch on its own
monolingual corpus. Run this notebook **twice**: once with `LANGUAGE = "hindi"`
(Model H) and once with `LANGUAGE = "nepali"` (Model L).

The two runs are fully independent, as the assignment requires: each attaches
only its own language's dataset, uses only that language's tokenizer and
vocabulary, writes to its own output directory, and is never initialised from
the other model's weights.

### Before running

1. **Accelerator:** Settings -> Accelerator -> **GPU P100**. The training code is
   single-GPU, so "T4 x2" would leave one card idle.
2. **Attach datasets** (Add Input):
   - `swaraj001/lma-code`
   - `swaraj001/lma-hindi-packed` *or* `swaraj001/lma-nepali-packed` - attach
     **only** the one matching `LANGUAGE`.
3. **Internet:** on, so a Devanagari font can be installed for heatmap labels.

### Session limits

Kaggle allows a 12-hour GPU session and roughly 30 GPU-hours per week.
`MAX_MINUTES` below stops training cleanly and writes a checkpoint before the
session is killed. If a run is cut short: save the notebook version, attach its
output as input to a new run, and set `RESUME_FROM` - training continues from the
exact optimizer, scheduler and step state.

## 1. Environment

In [ ]:
import os, sys, json, shutil, subprocess, time
from pathlib import Path

import torch

print("torch          :", torch.__version__)
print("CUDA available :", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU            :", torch.cuda.get_device_name(0))
    print("VRAM           : %.1f GB" % (props.total_memory / 1e9))
    print("compute cap    : %d.%d" % (props.major, props.minor))
    # bfloat16 needs Ampere (compute capability 8.0+). Kaggle's P100 (6.0) and
    # T4 (7.5) do not support it, so the trainer falls back to float16 with a
    # gradient scaler. Printing it here keeps that out of the surprises column.
    print("bf16 supported :", torch.cuda.is_bf16_supported())
    # PyTorch has dropped Pascal (sm_60). Kaggle's P100 is sm_60, so it can be
    # allocated and still be completely unusable: every kernel launch fails.
    # T4 is sm_75 and is supported, which is why the kernel requests
    # NvidiaTeslaT4 rather than the default accelerator.
    if (props.major, props.minor) < (7, 0):
        raise RuntimeError(
            "GPU %s is compute capability %d.%d, below the 7.0 minimum this "
            "PyTorch build supports. Request NvidiaTeslaT4 instead."
            % (torch.cuda.get_device_name(0), props.major, props.minor))
    # A real kernel launch, to prove the GPU is usable rather than merely present.
    _probe = (torch.ones(64, 64, device="cuda") @ torch.ones(64, 64, device="cuda")).sum()
    print("GPU matmul OK  :", float(_probe) == 64 * 64 * 64)
else:
    # Fail immediately rather than spending an 11-hour session on CPU, where
    # this run would need weeks. If the accelerator was requested but not
    # granted, the usual cause is that the Kaggle account is not phone-verified
    # (Settings -> Phone Verification), which gates both GPU and Internet.
    raise RuntimeError(
        "No GPU allocated. Requested accelerator was not granted - check "
        "Settings -> Accelerator, and that the account is phone-verified.")

## 2. Unpack the project code

In [ ]:
import zipfile

INPUT_ROOT = Path("/kaggle/input")
print("attached inputs:", sorted(p.name for p in INPUT_ROOT.iterdir()))

def find_code_source():
    # Kaggle decompresses archives uploaded to a dataset, and the mount
    # directory does not always match the dataset slug, so neither the path nor
    # the shape can be hardcoded. Search for whichever attached input actually
    # contains the two language trees, or a real zip holding them.
    for cand in sorted(INPUT_ROOT.iterdir()):
        if not cand.is_dir():
            continue
        if (cand / "hindi").is_dir() and (cand / "nepali").is_dir():
            return cand, None
        for z in sorted(cand.glob("*.zip")):
            if zipfile.is_zipfile(z):
                return cand, z
    raise FileNotFoundError(
        "no attached input contains the code tree; attach the lma-code dataset")

CODE_INPUT, archive = find_code_source()
REPO = Path("/kaggle/working/repo")

if REPO.exists():
    shutil.rmtree(REPO)
REPO.mkdir(parents=True)

if archive is not None:
    shutil.unpack_archive(archive, REPO)
    print("unpacked", archive.name, "from", CODE_INPUT)
else:
    for item in sorted(CODE_INPUT.iterdir()):
        target = REPO / item.name
        if item.is_dir():
            shutil.copytree(item, target)
        else:
            shutil.copy2(item, target)
    print("copied extracted tree from", CODE_INPUT)

# Put the repo root on the import path so `import hindi...` / `import nepali...`
# resolve, and make it the working directory so each package's REPO_ROOT-relative
# paths line up. Both language trees are present; this run touches only one.
sys.path.insert(0, str(REPO))
os.chdir(REPO)

print("repo at", REPO)
print(sorted(p.name for p in REPO.iterdir()))

## 3. Choose the language, and enforce corpus independence

In [ ]:
# ---- the only settings you change between the two runs ----
LANGUAGE = "hindi"          # "hindi" (Model H) or "nepali" (Model L)
MAX_MINUTES = 660           # stop cleanly at 11h, inside Kaggle's 12h cap
RESUME_FROM = None          # e.g. "/kaggle/input/<previous-output>/latest.pt"
# -----------------------------------------------------------

DATASETS = {"hindi": "/kaggle/input/lma-hindi-packed",
            "nepali": "/kaggle/input/lma-nepali-packed"}

def find_corpus(lang):
    # Falls back to searching for the mount that holds train.bin plus a
    # sidecar naming this language, since the mount name is not guaranteed to
    # match the dataset slug.
    expected = Path(DATASETS[lang])
    if expected.exists():
        return expected
    for cand in sorted(INPUT_ROOT.iterdir()):
        if cand.is_dir() and (cand / "train.bin").is_file():
            sidecar = cand / "train.json"
            if sidecar.is_file() and json.loads(sidecar.read_text())["language"] == lang:
                return cand
    raise FileNotFoundError("no attached input holds the " + lang + " corpus")

INPUT_DIR = find_corpus(LANGUAGE)
CONFIG    = REPO / (LANGUAGE + "/configs/model.yaml")
OUT_DIR   = Path("/kaggle/working/checkpoints/" + LANGUAGE)
# /kaggle/temp is scratch: it is NOT captured as kernel output. Staging the
# ~1GB token stream into /kaggle/working instead would push a gigabyte of
# regenerable data into the saved output for no reason.
DATA_DIR  = Path("/kaggle/temp/data/" + LANGUAGE)

assert CONFIG.exists(), CONFIG

# Guard the assignment's independence requirement: only this language's corpus
# may be mounted. Training Model H with Model L's data attached would be a
# violation even if the data went unused, so fail loudly rather than trust
# discipline.
other = "nepali" if LANGUAGE == "hindi" else "hindi"
if Path(DATASETS[other]).exists():
    raise RuntimeError(
        other + " dataset is also attached. Detach it: each model must be "
        "trained against its own corpus only.")

print("language :", LANGUAGE)
print("config   :", CONFIG)
print("input    :", INPUT_DIR)
print("files    :", sorted(p.name for p in INPUT_DIR.iterdir()))

## 4. Stage the token stream on local disk

`/kaggle/input` is a read-only network mount. Training samples random offsets into
the stream on every step, and random reads over that mount are slow, so the packed
`.bin` files are copied to local disk first. A one-time cost of a couple of minutes
that pays for itself many times over.

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
t0 = time.time()
for src in sorted(INPUT_DIR.glob("*")):
    if src.suffix in {".bin", ".json"}:
        dst = DATA_DIR / src.name
        if not dst.exists() or dst.stat().st_size != src.stat().st_size:
            shutil.copy2(src, dst)
            print("  staged %s (%.2f GB)" % (src.name, src.stat().st_size / 1e9))

# The tokenizer is needed for bits-per-byte and for decoding generated samples.
tok_dir = REPO / LANGUAGE / "tokenizer"
tok_dir.mkdir(parents=True, exist_ok=True)
for src in sorted(INPUT_DIR.glob("*.model")) + sorted(INPUT_DIR.glob("*.vocab")):
    shutil.copy2(src, tok_dir / src.name)

print("staged in %.0fs" % (time.time() - t0))
for j in sorted(DATA_DIR.glob("*.json")):
    m = json.loads(j.read_text())
    print("  %-5s %13s tokens  %9s docs  unk rate %.5f"
          % (m["split"], format(m["tokens"], ","), format(m["documents"], ","),
             m["unk_rate"]))

## 5. Verify the causal mask on this machine

The assignment asks for empirical proof that the model cannot see the future.
Running the check here, on the GPU that will do the training, means the number in
the report describes the actual training setup rather than a laptop.

In [ ]:
result = subprocess.run([sys.executable, LANGUAGE + "/tests/test_model.py"],
                        capture_output=True, text=True, cwd=REPO)
print(result.stdout[-3000:])
if result.returncode != 0:
    print(result.stderr[-3000:])
    raise RuntimeError("model tests failed - do not train on a broken model")

## 6. Parameter count and run plan

In [ ]:
# LANGUAGE is a runtime value, so the language's own package is imported
# dynamically. Model H and Model L have separate, independent code trees; this
# resolves to exactly one of them.
import importlib
cfg_mod = importlib.import_module(LANGUAGE + ".model.config")
gpt_mod = importlib.import_module(LANGUAGE + ".model.transformer")
RunConfig, GPT = cfg_mod.RunConfig, gpt_mod.GPT

run = RunConfig.load(CONFIG)
counts = GPT(run.model).parameter_count()
tps = run.train.tokens_per_step(run.model.block_size)

print("vocabulary        : %s" % format(run.model.vocab_size, ","))
print("layers/heads/dim  : %d/%d/%d" % (run.model.n_layer, run.model.n_head,
                                        run.model.n_embd))
print("context length    : %d" % run.model.block_size)
print("TOTAL parameters  : %s" % format(counts["total"], ","))
print("  embedding       : %s" % format(counts["embedding"], ","))
print("  non-embedding   : %s" % format(counts["non_embedding"], ","))
print("tokens/step       : %s" % format(tps, ","))
print("planned budget    : %.1fM tokens over %s steps"
      % (tps * run.train.max_steps / 1e6, format(run.train.max_steps, ",")))

## 7a. Smoke test: 50 steps on the real GPU

Kaggle's P100 and T4 have no bfloat16, so the trainer falls back to float16 with
a gradient scaler - a code path that cannot be exercised on a CPU-only machine.
Fifty steps take about a minute and confirm the mixed-precision path, the memory
footprint and the throughput before committing to a multi-hour run.

Check three things in the output below: the loss starts near `ln(vocab_size)`
(8.99 for Hindi, 9.21 for Nepali), it decreases, and it is never `nan`.
Then read the reported tok/s to predict the full run's wall-clock time.

In [ ]:
SMOKE_DIR = Path("/kaggle/working/smoke")
if SMOKE_DIR.exists():
    shutil.rmtree(SMOKE_DIR)

import math
smoke = subprocess.run(
    [sys.executable, "-u", "-m", LANGUAGE + ".train.train",
     "--config", str(CONFIG),
     "--data-dir", str(DATA_DIR),
     "--out-dir", str(SMOKE_DIR),
     "--max-steps", "50",
     "--no-resume"],
    cwd=REPO, capture_output=True, text=True)
print(smoke.stdout[-4000:])
if smoke.returncode != 0:
    print(smoke.stderr[-4000:])
    raise RuntimeError("smoke run failed - fix before spending GPU hours")

if "nan" in smoke.stdout.lower():
    raise RuntimeError("loss went to nan in the smoke run - check the fp16 path")

# Predict the full run from the observed throughput.
rates = [float(t.split("k tok/s")[0].split()[-1])
         for t in smoke.stdout.split("\n") if "k tok/s" in t]
if rates:
    tok_s = max(rates) * 1000
    total = tps * run.train.max_steps
    print("observed throughput : %.0fk tokens/s" % (tok_s / 1000))
    print("estimated full run  : %.1f hours for %s tokens"
          % (total / tok_s / 3600, format(total, ",")))
    print("session budget      : %.1f hours (MAX_MINUTES)" % (MAX_MINUTES / 60))

## 7b. Pretrain

In [ ]:
if RESUME_FROM:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(RESUME_FROM, OUT_DIR / "latest.pt")
    print("seeded resume from", RESUME_FROM)

cmd = [sys.executable, "-u", "-m", LANGUAGE + ".train.train",
       "--config", str(CONFIG),
       "--data-dir", str(DATA_DIR),
       "--out-dir", str(OUT_DIR),
       "--max-minutes", str(MAX_MINUTES)]
print(" ".join(cmd))
print()

proc = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print()
print("exit code:", proc.returncode)

## 8. Loss curves

In [ ]:
from IPython.display import Image, display
plot_loss_curves = importlib.import_module(LANGUAGE + ".eval.viz").plot_loss_curves

FIGDIR = Path("/kaggle/working/figures")
FIGDIR.mkdir(parents=True, exist_ok=True)
log = OUT_DIR / "training_log.jsonl"

plot_loss_curves({LANGUAGE: log}, FIGDIR / (LANGUAGE + "_loss_curve.png"),
                 x_axis="tokens")
plot_loss_curves({LANGUAGE: log}, FIGDIR / (LANGUAGE + "_loss_curve_by_step.png"),
                 x_axis="step")

display(Image(str(FIGDIR / (LANGUAGE + "_loss_curve.png"))))

## 9. Evaluation - perplexity, bits-per-byte, BLEU / chrF++ / ROUGE-L

Bits-per-byte is the number to compare across the two models: perplexity depends
on each language's own tokenizer, while BPB normalises by UTF-8 bytes of the
underlying text and so removes the tokenizer from the denominator.

In [ ]:
ckpt = OUT_DIR / "ckpt_best.pt"
if not ckpt.exists():
    ckpt = OUT_DIR / "latest.pt"

result = subprocess.run(
    [sys.executable, "-u", "-m", LANGUAGE + ".eval.evaluate",
     "--checkpoint", str(ckpt),
     "--data-dir", str(DATA_DIR),
     "--split", "test",
     "--out", "/kaggle/working/" + LANGUAGE + "_eval.json"],
    cwd=REPO, capture_output=True, text=True)
print(result.stdout[-6000:])
if result.returncode != 0:
    print(result.stderr[-4000:])

## 10. Attention analysis

In [ ]:
# Devanagari coverage for the heatmap tick labels; harmless if it fails.
subprocess.run("apt-get -qq install -y fonts-noto-devanagari > /dev/null 2>&1",
               shell=True)
import matplotlib.font_manager as fm
fm._load_fontmanager(try_read_cache=False)

eval_mod = importlib.import_module(LANGUAGE + ".eval.evaluate")
attn_mod = importlib.import_module(LANGUAGE + ".eval.attention")
viz_mod = importlib.import_module(LANGUAGE + ".eval.viz")
Evaluator = eval_mod.Evaluator
collect_attention = attn_mod.collect_attention
summarise_model_attention = attn_mod.summarise_model_attention
plot_attention_heatmap = viz_mod.plot_attention_heatmap
plot_attention_summary = viz_mod.plot_attention_summary

ev = Evaluator(ckpt, data_dir=DATA_DIR)
x, _ = ev.data.fixed_batches("test", 4, 1, seed=11)[0]
attentions = collect_attention(ev.model, x)
tokens = [ev.decode([int(t)]) for t in x[0].cpu().numpy()]

n_layer = len(attentions)
for layer in (0, n_layer - 1):          # one early layer, one late layer
    plot_attention_heatmap(
        attentions[layer][0].numpy(), tokens, layer, [0, 1, 2, 3],
        FIGDIR / ("%s_attention_layer%d.png" % (LANGUAGE, layer)),
        language=LANGUAGE)

summary = summarise_model_attention(ev.model, x)
plot_attention_summary({LANGUAGE: summary},
                       FIGDIR / (LANGUAGE + "_attention_summary.png"))

Path("/kaggle/working/" + LANGUAGE + "_attention.json").write_text(
    json.dumps(summary, indent=2))

display(Image(str(FIGDIR / (LANGUAGE + "_attention_layer0.png"))))
display(Image(str(FIGDIR / (LANGUAGE + "_attention_summary.png"))))

## 11. Bundle the artifacts

Two destinations, because the assignment treats them differently:

* **Checkpoints** are large binaries - download `checkpoints.zip`, upload it to
  **Google Drive**, and put the shareable link in the repository README with
  permissions set so graders can open it without requesting access.
* **Logs, metrics and figures** are small and must be **committed into the
  `phase-2` branch** under `report/phase2/`. The spec is explicit that graders
  will not open external links to find figures: if it is not in the branch, it is
  not graded.

In [ ]:
import zipfile

# Small artifacts -> commit these into report/phase2/ in the repo.
small = Path("/kaggle/working/report_phase2.zip")
with zipfile.ZipFile(small, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in FIGDIR.glob("*.png"):
        zf.write(p, "figures/" + p.name)
    for name in (LANGUAGE + "_eval.json", LANGUAGE + "_attention.json"):
        p = Path("/kaggle/working") / name
        if p.exists():
            zf.write(p, name)
    for name in ("training_log.jsonl", "config.yaml"):
        p = OUT_DIR / name
        if p.exists():
            zf.write(p, LANGUAGE + "_" + name)
print("report_phase2.zip  %.1f MB  -> commit into the repo"
      % (small.stat().st_size / 1e6))

# Large artifacts -> Google Drive, linked from the README.
big = Path("/kaggle/working/checkpoints.zip")
with zipfile.ZipFile(big, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in OUT_DIR.glob("*.pt"):
        zf.write(p, LANGUAGE + "/" + p.name)
print("checkpoints.zip    %.2f GB  -> Google Drive"
      % (big.stat().st_size / 1e9))